In [1]:
#!pip install torch_geometric_temporal
#!pip install networkx

import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import LabelEncoder
import seaborn as sns
import networkx as nx
import torch
from typing import Sequence, Union
from torch_geometric.data import Data
from torch_geometric_temporal.signal import DynamicGraphTemporalSignal
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric_temporal.nn.recurrent import TGCN
from torch_geometric_temporal.signal import temporal_signal_split
from torch_geometric_temporal.dataset import METRLADatasetLoader
from torch_geometric_temporal.signal import DynamicGraphTemporalSignalBatch
from torch_geometric.loader import DataLoader


# Set display option to show all rows 
pd.set_option('display.max_rows', None)

# Set display option to show all columns 
pd.set_option('display.max_columns', None)

In [2]:
correct_timestamps_df = pd.read_csv('final_df1.csv')
df = pd.read_csv('final_df.csv')
edges = pd.read_csv('edges_df.csv')

In [3]:
# Drop rows where 'Timestamp' is not in the list
df = df[df["TimeStamp"].isin(correct_timestamps_df['TimeStamp'].unique())]

In [4]:
df.head()

,TimeStamp,Compno_x,Compno_y,AverageMaximumTemperature_x,AverageMinimunTemperature_x,AverageMaximumRelativeHumidity_x,AverageMinimumRelativeHumidity_x,TotalRainfall_x,AverageTotalRadiation_x,AverageWindSpeed_x,AverageTotalRelativeEvaporation_x,Target_x,AverageMaximumTemperature_y,AverageMinimunTemperature_y,AverageMaximumRelativeHumidity_y,AverageMinimumRelativeHumidity_y,TotalRainfall_y,AverageTotalRadiation_y,AverageWindSpeed_y,AverageTotalRelativeEvaporation_y,Target_y,distance_km
696,2004-11-01,30142,30173,33.39,13.54,87.28,18.56,49.4,24.36,1.93,172.77,50.15375,34.69,18.15,69.96,17.32,2.4,30.37,1.64,196.68,46.40125,655.071776
697,2004-11-01,30142,30191,33.39,13.54,87.28,18.56,49.4,24.36,1.93,172.77,50.15375,35.38,14.80,74.53,12.48,5.4,30.63,2.07,221.20,49.56125,335.864713
698,2004-11-01,30142,30389,33.39,13.54,87.28,18.56,49.4,24.36,1.93,172.77,50.15375,33.89,14.20,85.20,16.46,11.6,30.83,1.96,208.88,50.37750,220.023323
699,2004-11-01,30142,30688,33.39,13.54,87.28,18.56,49.4,24.36,1.93,172.77,50.15375,33.79,16.17,49.00,11.45,8.8,28.59,2.72,143.41,36.74125,430.985911
700,2004-11-01,30142,30688,33.39,13.54,87.28,18.56,49.4,24.36,1.93,172.77,50.15375,33.79,16.17,49.00,11.45,8.8,28.59,2.72,143.41,36.74125,430.985911


In [5]:
# List of required nodes that should exist
required_nodes = {30688, 30142, 30173, 30191, 30389, 20272, 30692, 20197, 30899}

# Initialize a dictionary to store information
timestamp_info = {}

# Iterate through each unique timestamp
for ts in df['TimeStamp'].unique():
    # Filter dataframe for the current timestamp
    timestamp_data = df[df['TimeStamp'] == ts]
    # Gather all available nodes for the current timestamp
    available_nodes = list(set(timestamp_data['Compno_x']).union(set(timestamp_data['Compno_y'])))
    # Identify missing nodes
    missing_nodes = list(required_nodes.difference(available_nodes))
    # Store the information if there are missing nodes or just available nodes
    if missing_nodes:
        timestamp_info[ts] = {"available_nodes": available_nodes, "missing_nodes": missing_nodes}
    else:
        timestamp_info[ts] = {"available_nodes": available_nodes}

'''
# Print the result in the desired format
for ts, info in timestamp_info.items():
    if "missing_nodes" in info:  # Print only if there are missing nodes
        print(f"{ts}, {info['available_nodes']}, {info['missing_nodes']}")
    else:  # Print available nodes only
        print(f"{ts}, {info['available_nodes']}")
'''

'\n# Print the result in the desired format\nfor ts, info in timestamp_info.items():\n    if "missing_nodes" in info:  # Print only if there are missing nodes\n        print(f"{ts}, {info[\'available_nodes\']}, {info[\'missing_nodes\']}")\n    else:  # Print available nodes only\n        print(f"{ts}, {info[\'available_nodes\']}")\n'

In [6]:
edges.head(5)

,CompNo1,CompNo2,distance_km
0,30688,30142,430.985911
1,30688,30173,399.409471
2,30688,30191,156.097747
3,30688,30389,294.369313
4,30688,20272,156.716859


In [7]:
print(edges['CompNo1'].unique())

[30688 30142 30173 30191 30389 20272 30692 20197 30899]


In [8]:
 # Define propagate_features function
def propagate_features(graph):
    for node in graph.nodes():
        if graph.nodes[node]["feature"] is None:  # Check for missing features
            neighbors = list(graph.neighbors(node))
            neighbor_features = [graph.nodes[neighbor]["feature"] for neighbor in neighbors if graph.nodes[neighbor]["feature"] is not None]
                
            if neighbor_features:
                # Average the features of the neighbors
                propagated_feature = np.mean(neighbor_features, axis=0)
                graph.nodes[node]["feature"] = propagated_feature
                #print(f"Features for node {node} have been filled: {propagated_feature}")
            #else:
                #print(f"Node {node} has no neighbors with features to propagate from.")
    return graph
    

In [9]:
corrected_df = pd.DataFrame(columns=[
    'TimeStamp', 'Compno_x', 'Compno_y', 'distance_km', 'AverageMaximumTemperature_x', 
    'AverageMinimunTemperature_x', 'AverageMaximumRelativeHumidity_x', 
    'AverageMinimumRelativeHumidity_x', 'TotalRainfall_x', 'AverageTotalRadiation_x', 
    'AverageWindSpeed_x', 'AverageTotalRelativeEvaporation_x', 'Target_x', 
    'AverageMaximumTemperature_y', 'AverageMinimunTemperature_y', 
    'AverageMaximumRelativeHumidity_y', 'AverageMinimumRelativeHumidity_y', 
    'TotalRainfall_y', 'AverageTotalRadiation_y', 'AverageWindSpeed_y', 
    'AverageTotalRelativeEvaporation_y', 'Target_y'
])


# Iterate through each unique timestamp
for timestamp in df['TimeStamp'].unique():
    #print(f"\nProcessing graph for timestamp: {timestamp}")
    #print('========================================================================')
    
    # Filter dataframe for the current timestamp
    timestamp_data = df[df['TimeStamp'] == timestamp]
      
    # Initialize the graph
    G = nx.Graph()
    
    # Add nodes from Compno_x and Compno_y with features
    for index, row in timestamp_data.iterrows():
        # Add Compno_x with features
        G.add_node(row['Compno_x'], 
                   feature=np.array([
                       row['AverageMaximumTemperature_x'], row['AverageMinimunTemperature_x'], 
                       row['AverageMaximumRelativeHumidity_x'], row['AverageMinimumRelativeHumidity_x'], 
                       row['TotalRainfall_x'], row['AverageTotalRadiation_x'], 
                       row['AverageWindSpeed_x'], row['AverageTotalRelativeEvaporation_x'], 
                       row['Target_x']]) if not pd.isna(row['AverageMaximumTemperature_x']) else None)
        
        # Add Compno_y with all features
        G.add_node(row['Compno_y'], 
                   feature=np.array([
                       row['AverageMaximumTemperature_y'], row['AverageMinimunTemperature_y'], 
                       row['AverageMaximumRelativeHumidity_y'], row['AverageMinimumRelativeHumidity_y'], 
                       row['TotalRainfall_y'], row['AverageTotalRadiation_y'], 
                       row['AverageWindSpeed_y'], row['AverageTotalRelativeEvaporation_y'], 
                       row['Target_y']]) if not pd.isna(row['AverageMaximumTemperature_y']) else None)
        
        # Add edge between Compno_x and Compno_y
        G.add_edge(row['Compno_x'], row['Compno_y'], distance=row['distance_km'])
        G.add_edge(row['Compno_y'], row['Compno_x'], distance=row['distance_km'])
    
    # Add any missing nodes from the predefined list, setting feature=None
    for node in required_nodes:
        if node not in G.nodes:
            G.add_node(node, feature=None)
            
    # Add edges for missing nodes to every other node in the graph
    for missing_node in required_nodes:
        if missing_node in G.nodes and G.nodes[missing_node]['feature'] is None:  # Ensure it's a missing node
            for existing_node in G.nodes:
                if missing_node != existing_node:  # Prevent self-loops
                    #G.add_edge(missing_node, existing_node, distance=np.nan)  # Set distance to NaN if unavailable
                    
                    # Retrieve distance from the edges dataframe
                    edge_row = edges[((edges['CompNo1'] == missing_node) & (edges['CompNo2'] == existing_node)) | 
                                     ((edges['CompNo1'] == existing_node) & (edges['CompNo2'] == missing_node))]
                    if not edge_row.empty:
                        distance = edge_row.iloc[0]['distance_km']
                    else:
                        distance = np.nan  # If distance is not available
                    
                    # Add the edge with the retrieved distance
                    G.add_edge(missing_node, existing_node, distance=distance)

    '''
    print('===========================================================================')
    # Print node features
    print("Nodes and their features:")
    for node, data in G.nodes(data=True):
        print(f"Node {node}: Features {data['feature']}")
    
    # Print edges and their distances
    print("Edges and their distances:")
    for edge in G.edges(data=True):
        node1, node2, data = edge
        print(f"Edge ({node1}, {node2}): Distance {data['distance']} km")
    '''

    # Apply the propagation function
    G = propagate_features(G)
    
    '''
    print('==================================================================================')
     # Print node features
    print("Nodes and their features:")
    for node, data in G.nodes(data=True):
        print(f"Node {node}: Features {data['feature']}")
    
    # Print edges and their distances
    print("Edges and their distances:")
    for edge in G.edges(data=True):
        node1, node2, data = edge
        print(f"Edge ({node1}, {node2}): Distance {data['distance']} km")
    '''
       
    #print('===================================================================================')

        # Create a dataframe for nodes
    node_data = []
    for node, data in G.nodes(data=True):
        node_data.append({
            'Node': node,
            'AverageMaximumTemperature': data['feature'][0], # Extract the feature
            'AverageMinimunTemperature': data['feature'][1], # Extract the feature
            'AverageMaximumRelativeHumidity': data['feature'][2], # Extract the feature
            'AverageMinimumRelativeHumidity': data['feature'][3], # Extract the feature
            'TotalRainfall': data['feature'][4], # Extract the feature
            'AverageTotalRadiation': data['feature'][5], # Extract the feature
            'AverageWindSpeed': data['feature'][6], # Extract the feature
            'AverageTotalRelativeEvaporation': data['feature'][7], # Extract the feature
            'Target': data['feature'][8] # Extract the feature
        })
    nodes_df = pd.DataFrame(node_data)

    #print(nodes_df)
    #print()
    
    # Create a dataframe for edges
    edge_data = []
    for node1, node2, data in G.edges(data=True):
        edge_data.append({
            'Node1': node1,
            'Node2': node2,
            'Distance': data.get('distance', None)  # Extract the distance
        })
        
        '''
        print('================================================================')
        print(nodes_df[nodes_df['Node'] == node1]['AverageMaximumTemperature'].values[0])
        print(nodes_df[nodes_df['Node'] == node2]['AverageMaximumTemperature'].values[0])
        print('================================================================')
        '''
        
        # Create a DataFrame with the new row
        new_row_df = pd.DataFrame([{
            'TimeStamp': timestamp,
            'Compno_x': node1, 
            'Compno_y': node2,
            'distance_km': data.get('distance', None),
            'AverageMaximumTemperature_x': (nodes_df[nodes_df['Node'] == node1]['AverageMaximumTemperature'].values[0]),
            'AverageMinimunTemperature_x': (nodes_df[nodes_df['Node'] == node1]['AverageMaximumTemperature'].values[0]),
            'AverageMaximumRelativeHumidity_x': (nodes_df[nodes_df['Node'] == node1]['AverageMaximumRelativeHumidity'].values[0]),
            'AverageMinimumRelativeHumidity_x': (nodes_df[nodes_df['Node'] == node1]['AverageMinimumRelativeHumidity'].values[0]),
            'TotalRainfall_x': (nodes_df[nodes_df['Node'] == node1]['TotalRainfall'].values[0]),
            'AverageTotalRadiation_x': (nodes_df[nodes_df['Node'] == node1]['AverageTotalRadiation'].values[0]),
            'AverageWindSpeed_x': (nodes_df[nodes_df['Node'] == node1]['AverageWindSpeed'].values[0]),
            'AverageTotalRelativeEvaporation_x': (nodes_df[nodes_df['Node'] == node1]['AverageTotalRelativeEvaporation'].values[0]),
            'Target_x': (nodes_df[nodes_df['Node'] == node1]['Target'].values[0]),
            'AverageMaximumTemperature_y': (nodes_df[nodes_df['Node'] == node2]['AverageMaximumTemperature'].values[0]),
            'AverageMinimunTemperature_y': (nodes_df[nodes_df['Node'] == node2]['AverageMinimunTemperature'].values[0]),
            'AverageMaximumRelativeHumidity_y': (nodes_df[nodes_df['Node'] == node2]['AverageMaximumRelativeHumidity'].values[0]),
            'AverageMinimumRelativeHumidity_y': (nodes_df[nodes_df['Node'] == node2]['AverageMinimumRelativeHumidity'].values[0]),
            'TotalRainfall_y': (nodes_df[nodes_df['Node'] == node2]['TotalRainfall'].values[0]),
            'AverageTotalRadiation_y': (nodes_df[nodes_df['Node'] == node2]['AverageTotalRadiation'].values[0]),
            'AverageWindSpeed_y': (nodes_df[nodes_df['Node'] == node2]['AverageWindSpeed'].values[0]),
            'AverageTotalRelativeEvaporation_y': (nodes_df[nodes_df['Node'] == node2]['AverageTotalRelativeEvaporation'].values[0]),
            'Target_y': (nodes_df[nodes_df['Node'] == node2]['Target'].values[0])
        }])
        
        # Concatenate the new row with the original DataFrame
        corrected_df = pd.concat([corrected_df, new_row_df], ignore_index=True)
        
        # Create a DataFrame with the new row
        new_row_df = pd.DataFrame([{
            'TimeStamp': timestamp,
            'Compno_x': node2, 
            'Compno_y': node1,
            'distance_km': data.get('distance', None),
            'AverageMaximumTemperature_x': (nodes_df[nodes_df['Node'] == node2]['AverageMaximumTemperature'].values[0]),
            'AverageMinimunTemperature_x': (nodes_df[nodes_df['Node'] == node2]['AverageMaximumTemperature'].values[0]),
            'AverageMaximumRelativeHumidity_x': (nodes_df[nodes_df['Node'] == node2]['AverageMaximumRelativeHumidity'].values[0]),
            'AverageMinimumRelativeHumidity_x': (nodes_df[nodes_df['Node'] == node2]['AverageMinimumRelativeHumidity'].values[0]),
            'TotalRainfall_x': (nodes_df[nodes_df['Node'] == node2]['TotalRainfall'].values[0]),
            'AverageTotalRadiation_x': (nodes_df[nodes_df['Node'] == node2]['AverageTotalRadiation'].values[0]),
            'AverageWindSpeed_x': (nodes_df[nodes_df['Node'] == node2]['AverageWindSpeed'].values[0]),
            'AverageTotalRelativeEvaporation_x': (nodes_df[nodes_df['Node'] == node2]['AverageTotalRelativeEvaporation'].values[0]),
            'Target_x': (nodes_df[nodes_df['Node'] == node2]['Target'].values[0]),
            'AverageMaximumTemperature_y': (nodes_df[nodes_df['Node'] == node1]['AverageMaximumTemperature'].values[0]),
            'AverageMinimunTemperature_y': (nodes_df[nodes_df['Node'] == node1]['AverageMinimunTemperature'].values[0]),
            'AverageMaximumRelativeHumidity_y': (nodes_df[nodes_df['Node'] == node1]['AverageMaximumRelativeHumidity'].values[0]),
            'AverageMinimumRelativeHumidity_y': (nodes_df[nodes_df['Node'] == node1]['AverageMinimumRelativeHumidity'].values[0]),
            'TotalRainfall_y': (nodes_df[nodes_df['Node'] == node1]['TotalRainfall'].values[0]),
            'AverageTotalRadiation_y': (nodes_df[nodes_df['Node'] == node1]['AverageTotalRadiation'].values[0]),
            'AverageWindSpeed_y': (nodes_df[nodes_df['Node'] == node1]['AverageWindSpeed'].values[0]),
            'AverageTotalRelativeEvaporation_y': (nodes_df[nodes_df['Node'] == node1]['AverageTotalRelativeEvaporation'].values[0]),
            'Target_y': (nodes_df[nodes_df['Node'] == node1]['Target'].values[0])
        }])

        # Concatenate the new row with the original DataFrame
        corrected_df = pd.concat([corrected_df, new_row_df], ignore_index=True)

        #print(corrected_df)

In [10]:
corrected_df.head(5)

,TimeStamp,Compno_x,Compno_y,distance_km,AverageMaximumTemperature_x,AverageMinimunTemperature_x,AverageMaximumRelativeHumidity_x,AverageMinimumRelativeHumidity_x,TotalRainfall_x,AverageTotalRadiation_x,AverageWindSpeed_x,AverageTotalRelativeEvaporation_x,Target_x,AverageMaximumTemperature_y,AverageMinimunTemperature_y,AverageMaximumRelativeHumidity_y,AverageMinimumRelativeHumidity_y,TotalRainfall_y,AverageTotalRadiation_y,AverageWindSpeed_y,AverageTotalRelativeEvaporation_y,Target_y
0,2004-11-01,30142,30173,655.071776,33.39,33.39,87.28,18.56,49.4,24.36,1.93,172.77,50.15375,34.69,18.15,69.96,17.32,2.4,30.37,1.64,196.68,46.40125
1,2004-11-01,30173,30142,655.071776,34.69,34.69,69.96,17.32,2.4,30.37,1.64,196.68,46.40125,33.39,13.54,87.28,18.56,49.4,24.36,1.93,172.77,50.15375
2,2004-11-01,30142,30191,335.864713,33.39,33.39,87.28,18.56,49.4,24.36,1.93,172.77,50.15375,35.38,14.80,74.53,12.48,5.4,30.63,2.07,221.20,49.56125
3,2004-11-01,30191,30142,335.864713,35.38,35.38,74.53,12.48,5.4,30.63,2.07,221.20,49.56125,33.39,13.54,87.28,18.56,49.4,24.36,1.93,172.77,50.15375
4,2004-11-01,30142,30389,220.023323,33.39,33.39,87.28,18.56,49.4,24.36,1.93,172.77,50.15375,33.89,14.20,85.20,16.46,11.6,30.83,1.96,208.88,50.37750


In [11]:
# Custom encoding dictionary
encoding_dict = {30688: 0, 30142: 1, 30173: 2, 30191: 3, 30389: 4, 30692: 5, 30899: 6,  20272: 7, 20197: 8} #'São Paulo': 1, 'München': 2, '東京': 3}

# Apply custom encoding
corrected_df['Compno_x'] = corrected_df['Compno_x'].map(encoding_dict)
corrected_df['Compno_y'] = corrected_df['Compno_y'].map(encoding_dict)

In [12]:
corrected_df = corrected_df.sort_values(by=['TimeStamp', 'Compno_x', 'Compno_y'])

In [15]:
corrected_df.head(200)

,TimeStamp,Compno_x,Compno_y,distance_km,AverageMaximumTemperature_x,AverageMinimunTemperature_x,AverageMaximumRelativeHumidity_x,AverageMinimumRelativeHumidity_x,TotalRainfall_x,AverageTotalRadiation_x,AverageWindSpeed_x,AverageTotalRelativeEvaporation_x,Target_x,AverageMaximumTemperature_y,AverageMinimunTemperature_y,AverageMaximumRelativeHumidity_y,AverageMinimumRelativeHumidity_y,TotalRainfall_y,AverageTotalRadiation_y,AverageWindSpeed_y,AverageTotalRelativeEvaporation_y,Target_y
7,2004-11-01,0,1,430.985911,33.790000,33.790000,49.000000,11.450000,8.800000,28.590000,2.720000,143.410000,36.741250,33.390000,13.540000,87.280000,18.560000,49.400000,24.360000,1.930000,172.770000,50.153750
21,2004-11-01,0,2,399.409471,33.790000,33.790000,49.000000,11.450000,8.800000,28.590000,2.720000,143.410000,36.741250,34.690000,18.150000,69.960000,17.320000,2.400000,30.370000,1.640000,196.680000,46.401250
33,2004-11-01,0,3,156.097747,33.790000,33.790000,49.000000,11.450000,8.800000,28.590000,2.720000,143.410000,36.741250,35.380000,14.800000,74.530000,12.480000,5.400000,30.630000,2.070000,221.200000,49.561250
43,2004-11-01,0,4,294.369313,33.790000,33.790000,49.000000,11.450000,8.800000,28.590000,2.720000,143.410000,36.741250,33.890000,14.200000,85.200000,16.460000,11.600000,30.830000,1.960000,208.880000,50.377500
52,2004-11-01,0,5,146.844184,33.790000,33.790000,49.000000,11.450000,8.800000,28.590000,2.720000,143.410000,36.741250,32.780000,14.630000,62.340000,13.020000,2.400000,28.860000,3.500000,134.030000,36.445000
58,2004-11-01,0,6,369.437300,33.790000,33.790000,49.000000,11.450000,8.800000,28.590000,2.720000,143.410000,36.741250,33.986667,15.248333,71.385000,14.881667,13.333333,28.940000,2.303333,179.495000,44.946667
56,2004-11-01,0,7,156.716859,33.790000,33.790000,49.000000,11.450000,8.800000,28.590000,2.720000,143.410000,36.741250,33.986667,15.248333,71.385000,14.881667,13.333333,28.940000,2.303333,179.495000,44.946667
54,2004-11-01,0,8,37.742541,33.790000,33.790000,49.000000,11.450000,8.800000,28.590000,2.720000,143.410000,36.741250,33.986667,15.248333,71.385000,14.881667,13.333333,28.940000,2.303333,179.495000,44.946667
6,2004-11-01,1,0,430.985911,33.390000,33.390000,87.280000,18.560000,49.400000,24.360000,1.930000,172.770000,50.153750,33.790000,16.170000,49.000000,11.450000,8.800000,28.590000,2.720000,143.410000,36.741250
0,2004-11-01,1,2,655.071776,33.390000,33.390000,87.280000,18.560000,49.400000,24.360000,1.930000,172.770000,50.153750,34.690000,18.150000,69.960000,17.320000,2.400000,30.370000,1.640000,196.680000,46.401250


In [16]:
corrected_df.to_csv('corrected_df.csv', index=False)